In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# 1. API key (get it from openweathermap.org)
API_KEY = "b0db8b68211990ca311f0f86d3906995"



In [12]:
# 2. Get forecast data for three cities
cities = {
    "Pune": (18.5204, 73.8567),
    "Mumbai": (19.0760, 72.8777),
    "Delhi": (28.6139, 77.2090)
}


In [14]:
url = "https://api.openweathermap.org/data/2.5/forecast"
rows = []

for city, (lat, lon) in cities.items():
    r = requests.get(url, params={
        "lat": lat, "lon": lon,
        "appid": API_KEY, "units": "metric"
    }, timeout=20)
    r.raise_for_status()
    data = r.json()

    for w in data["list"]:
        rows.append([
            city, lat, lon,
            pd.to_datetime(w["dt"], unit="s", utc=True),
            w["main"].get("temp"),
            w["main"].get("humidity"),
            w.get("wind", {}).get("speed"),
            w.get("rain", {}).get("3h", 0)
        ])


HTTPError: 401 Client Error: Unauthorized for url: https://api.openweathermap.org/data/2.5/forecast?lat=18.5204&lon=73.8567&appid=133951f9c48b2c3e747a7ce8eadfde4c&units=metric

In [ ]:
# 3. Create and clean the DataFrame
df = pd.DataFrame(rows, columns=[
    "City", "Latitude", "Longitude", "Time",
    "Temperature", "Humidity", "WindSpeed", "Precipitation"
])

df["Time"] = df["Time"].dt.tz_convert("Asia/Kolkata")
cols = ["Temperature", "Humidity", "WindSpeed", "Precipitation"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
df["Precipitation"] = df["Precipitation"].fillna(0)
df = df.dropna(subset=["Temperature", "Humidity", "WindSpeed"])

In [ ]:

# 4 & 5. Summary statistics: mean, minimum and maximum
print(df.groupby("City")[cols].agg(["mean", "min", "max"]).round(2))


In [ ]:
# 6. Temperature changes over time
for city, d in df.groupby("City"):
    plt.plot(d["Time"], d["Temperature"], label=city)

plt.xlabel("Time")
plt.ylabel("Temperature (°C)")
plt.title("Temperature Forecast")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Plot precipitation and wind speed for Pune
pune = df[df["City"] == "Pune"]
pune.plot(x="Time", y=["Precipitation", "WindSpeed"],
          subplots=True, marker="o", figsize=(8, 5))
plt.tight_layout()
plt.show()


In [ ]:
# 7. Daily weather averages
df["Date"] = df["Time"].dt.date
print("\nDaily Averages:\n",
      df.groupby(["City", "Date"])[cols].mean().round(2))

In [ ]:
# 8. Simple geographical visualization
geo = df.groupby("City").agg(
    Latitude=("Latitude", "first"),
    Longitude=("Longitude", "first"),
    Temperature=("Temperature", "mean")
)

plt.scatter(geo["Longitude"], geo["Latitude"],
            c=geo["Temperature"], s=150, cmap="coolwarm")
for city, row in geo.iterrows():
    plt.text(row["Longitude"], row["Latitude"], city)

plt.colorbar(label="Average Temperature (°C)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Weather by Location")
plt.show()


In [ ]:
# 9. Correlation between weather attributes
sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Weather Correlation")
plt.show()